### 1. Leer tablas Bronze

In [2]:
customer = spark.table("bronze.customer")
customer_address = spark.table("bronze.customeraddress")
address = spark.table("bronze.address")

product = spark.table("bronze.product")
category = spark.table("bronze.productcategory")
model = spark.table("bronze.productmodel")

header = spark.table("bronze.salesorderheader")
detail = spark.table("bronze.salesorderdetail")

StatementMeta(SparkProyecto, 3, 2, Finished, Available, Finished, False)

### 2. Crear dimensión Cliente (Silver)

In [3]:
dim_customer = (
    customer
    .join(customer_address, "CustomerID", "left")
    .join(address, "AddressID", "left")
    .select(
        "CustomerID",
        "AddressID",
        "AddressLine1",
        "City",
        "StateProvince"
    )
)

StatementMeta(SparkProyecto, 3, 3, Finished, Available, Finished, False)

In [4]:
dim_customer.write \
    .mode("overwrite") \
    .saveAsTable("silver.dim_customer")

StatementMeta(SparkProyecto, 3, 4, Finished, Available, Finished, False)

In [1]:
dim_customer.write \
    .mode("overwrite") \
    .parquet(
        "abfss://adlsdatamau@sadp203mau.dfs.core.windows.net/silver/dim_customer"
    )

StatementMeta(, , -1, Cancelled, , Cancelled, True)

### 3. Crear dimensión Producto

In [8]:
from pyspark.sql.functions import col

dim_product = (
    product.alias("p")
    .join(category.alias("c"), "ProductCategoryID", "left")
    .join(model.alias("m"), "ProductModelID", "left")
    .select(
        col("p.ProductID"),
        col("p.Name").alias("ProductName"),
        col("p.ProductNumber"),
        col("p.StandardCost"),
        col("p.ListPrice"),
        col("c.Name").alias("CategoryName"),
        col("m.Name").alias("ModelName")
    )
)

StatementMeta(SparkProyecto, 3, 8, Finished, Available, Finished, False)

In [9]:
dim_product.write \
    .mode("overwrite") \
    .saveAsTable("silver.dim_product")

StatementMeta(SparkProyecto, 3, 9, Finished, Available, Finished, False)

In [ ]:
dim_product.write \
    .mode("overwrite") \
    .parquet(
        "abfss://adlsdatamau@sadp203mau.dfs.core.windows.net/silver/dim_product"
    )

### 4. Crear Fact Ventas

In [10]:
fact_sales = (
    detail.alias("d")
    .join(
        header.alias("h"),
        "SalesOrderID"
    )
    .select(
        "SalesOrderID",
        "SalesOrderDetailID",
        "OrderQty",
        "ProductID",
        "CustomerID",
        "OrderDate",
        "UnitPrice",
        "LineTotal"
    )
)

StatementMeta(SparkProyecto, 3, 10, Finished, Available, Finished, False)

In [11]:
fact_sales.write \
    .mode("overwrite") \
    .saveAsTable("silver.fact_sales")

StatementMeta(SparkProyecto, 3, 11, Finished, Available, Finished, False)

In [ ]:
fact_sales.write \
    .mode("overwrite") \
    .parquet(
        "abfss://adlsdatamau@sadp203mau.dfs.core.windows.net/silver/fact_sales"
    )